# HW12 Part B :: Semantic Segmentation in Videos

COSC 6373 - Adam Nelson-Archer, 2140122

## Setup

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from urllib.request import Request, urlopen
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)

## Question 1

Are image segmentation models considered multi-task architectures, and what does "multi-task" mean in this context?

[Your answer goes here. Provide a brief explanation.]

## Question 2 & 3 - Video Segmentation

Use a short, low-resolution video (5-10 seconds, 240x480 pixels) and perform semantic segmentation using DeepLabV3 and Lite R-ASPP with MobileNetV3 weights.

In [ ]:
# Load video
REPO_ROOT = Path("../..")
VIDEO_NAME = "sample_video.mp4"
VIDEO_PATH = REPO_ROOT / VIDEO_NAME

if not VIDEO_PATH.exists():
    # Download a sample low-res video if not present
    video_url = "https://download.samplelib.com/mp4/sample-5s.mp4"
    req = Request(video_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urlopen(req) as response, open(VIDEO_PATH, 'wb') as out_file:
        out_file.write(response.read())
    print(f"Downloaded sample video to {VIDEO_PATH.resolve()}")
else:
    print(f"Video already exists at {VIDEO_PATH.resolve()}")

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load models
deeplab = models.segmentation.deeplabv3_mobilenet_v3_large(weights='DEFAULT').to(device).eval()
lraspp = models.segmentation.lraspp_mobilenet_v3_large(weights='DEFAULT').to(device).eval()

# Preprocessing transforms
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((240, 480)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Code to parse video frames and apply the models
cap = cv2.VideoCapture(str(VIDEO_PATH))

frames_processed = 0
max_frames = 10  # Only processing a few frames for display purposes in the notebook

fig, axes = plt.subplots(max_frames, 3, figsize=(15, 4 * max_frames))
axes[0, 0].set_title("Original Frame")
axes[0, 1].set_title("DeepLabV3")
axes[0, 2].set_title("Lite R-ASPP")

with torch.no_grad():
    while cap.isOpened() and frames_processed < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (480, 240))
        
        # Prepare input tensor
        input_tensor = preprocess(frame_resized).unsqueeze(0).to(device)
        
        # Inference: DeepLabV3
        out_dl = deeplab(input_tensor)['out'][0]
        pred_dl = out_dl.argmax(0).byte().cpu().numpy()
        
        # Inference: LR-ASPP
        out_lr = lraspp(input_tensor)['out'][0]
        pred_lr = out_lr.argmax(0).byte().cpu().numpy()
        
        # Display
        axes[frames_processed, 0].imshow(frame_resized)
        axes[frames_processed, 0].axis("off")
        
        axes[frames_processed, 1].imshow(pred_dl, cmap='tab20')
        axes[frames_processed, 1].axis("off")
        
        axes[frames_processed, 2].imshow(pred_lr, cmap='tab20')
        axes[frames_processed, 2].axis("off")
        
        frames_processed += 1

cap.release()
plt.tight_layout()
plt.show()

## Question 6

Which model seems to perform better? (qualitative comparison via visual inspection)

[Provide your qualitative comparison here based on the outputs above.]

## Question 7

What is the main difference between DeepLabV3 and LR-ASPP? Please elaborate.

[Elaborate on the architectural differences here.]

## Acknowledgment

I used a GPT-5.3-Codex to help scaffold and organize this notebook.

Gemini-3.1 was used to check the result and validate conformity with the assignment outline.

All observations and responses were written by me, Adam Nelson-Archer.